# 11a — Demographic Baseline Models

This notebook evaluates baseline classifiers using the cleaned demographic dataset produced by `02a_demographic_preparation.ipynb`.

## Objective

The notebook:

- loads `data/processed/demographics_clean.csv`;
- selects only demographic predictors;
- creates reproducible stratified train, validation, and test subsets from the demographic dataset;
- uses the shared preprocessing and modeling framework in `src/modeling`;
- evaluates the Dummy Classifier, multinomial Logistic Regression, and Random Forest;
- uses the project-configured class weighting for Logistic Regression and Random Forest;
- reports validation and test metrics, classification reports, and confusion matrices;
- saves confusion-matrix figures using the shared output utilities;
- saves demographic-specific outputs using `src/modeling/outputs.py`;
- provides initial observations from the model comparison results.

The shared model training and evaluation logic is not rewritten in this notebook.

## 1. Libraries and project paths

In [1]:
from pathlib import Path
import sys
import pandas as pd

from sklearn.model_selection import train_test_split

pd.set_option("display.max_rows", 100)
pd.set_option("display.max_columns", 100)

# Locate the project root whether the notebook is run from the
# repository root or from the notebooks directory.
cwd = Path.cwd().resolve()

if (cwd / "src").exists():
    PROJECT_ROOT = cwd
elif (cwd.parent / "src").exists():
    PROJECT_ROOT = cwd.parent
else:
    raise FileNotFoundError(
        "Could not locate the project root containing the src directory."
    )

if str(PROJECT_ROOT) not in sys.path:
    sys.path.append(str(PROJECT_ROOT))

DEMOGRAPHIC_FILE = (
    PROJECT_ROOT
    / "data"
    / "processed"
    / "demographics_clean.csv"
)

print(f"Project root: {PROJECT_ROOT}")
print(f"Demographic file: {DEMOGRAPHIC_FILE}")

Project root: C:\Users\bramw\OneDrive - GUSCanada\Desktop\Semester 5\Capstone Project\AI-Assisted-Screening-of-Parkinson-s-Disease
Demographic file: C:\Users\bramw\OneDrive - GUSCanada\Desktop\Semester 5\Capstone Project\AI-Assisted-Screening-of-Parkinson-s-Disease\data\processed\demographics_clean.csv


## 2. Import the shared modeling framework

In [2]:
from src.modeling.preprocessing import (
    prepare_dataset,
    identify_feature_types,
)

from src.modeling.baseline import get_dummy_classifier

from src.modeling.models import (
    get_logistic_regression,
    get_random_forest,
)

from src.modeling.workflow import run_models

from src.modeling.outputs import (
    save_metrics,
    save_classification_report,
    save_confusion_matrix,
    save_model_comparison,
    save_table,
    save_confusion_matrix_figure,
)

from src.modeling.config import (
    TARGET_COLUMN,
    TRAIN_SIZE,
    VALIDATION_SIZE,
    TEST_SIZE,
    RANDOM_STATE,
    CLASS_WEIGHT,
    USE_CLASS_WEIGHT,
    PRIMARY_METRIC,
)

## 3. Load the cleaned demographic dataset

In [3]:
assert DEMOGRAPHIC_FILE.exists(), (
    f"Clean demographic dataset not found: {DEMOGRAPHIC_FILE}\n"
    "Run 02a_demographic_preparation.ipynb first."
)

demographics = pd.read_csv(
    DEMOGRAPHIC_FILE,
    dtype={"patient_id": str},
)

print(f"Dataset shape: {demographics.shape}")
print(f"Unique participants: {demographics['patient_id'].nunique():,}")
display(demographics.head())

Dataset shape: (469, 15)
Unique participants: 469


,patient_id,study_id,condition_original,condition_group,label,age,age_at_diagnosis,height_cm,weight_kg,gender,handedness,family_history_any,family_history_first_degree,alcohol_effect_on_tremor,duplicate_patient_id
0,1,PADS,Healthy,Healthy Control,0,56.0,NaN,173.0,78.0,Male,Right,Yes,Yes,Unknown,False
1,2,PADS,Other Movement Disorders,Other Movement Disorder,2,81.0,69.0,193.0,104.0,Male,Right,No,Unknown,No Effect,False
2,3,PADS,Healthy,Healthy Control,0,45.0,NaN,170.0,78.0,Female,Right,No,Unknown,Unknown,False
3,4,PADS,Parkinson's,Parkinson's Disease,1,67.0,63.0,161.0,90.0,Female,Right,No,Unknown,No Effect,False
4,5,PADS,Parkinson's,Parkinson's Disease,1,75.0,65.0,172.0,86.0,Male,Left,No,Unknown,Unknown,False


`demographics_clean.csv` is the direct output of the demographic preparation notebook and is the sole modeling data source used here.

## 4. Select demographic predictors

In [4]:
demographic_features = [
    "age",
    "age_at_diagnosis",
    "height_cm",
    "weight_kg",
    "gender",
    "handedness",
    "family_history_any",
    "family_history_first_degree",
    "alcohol_effect_on_tremor",
]

framework_columns = [
    "patient_id",
    "study_id",
    "duplicate_patient_id",
    "condition_original",
    "condition_group",
    TARGET_COLUMN,
]

required_columns = [
    "patient_id",
    TARGET_COLUMN,
    *demographic_features,
]

missing_columns = [
    column
    for column in required_columns
    if column not in demographics.columns
]

assert not missing_columns, (
    f"Missing required demographic columns: {missing_columns}"
)

columns_to_keep = [
    column
    for column in framework_columns + demographic_features
    if column in demographics.columns
]

demo_df = demographics[columns_to_keep].copy()

print("Columns retained:")
print(demo_df.columns.tolist())
print(f"\nModeling table shape: {demo_df.shape}")

Columns retained:
['patient_id', 'study_id', 'duplicate_patient_id', 'condition_original', 'condition_group', 'label', 'age', 'age_at_diagnosis', 'height_cm', 'weight_kg', 'gender', 'handedness', 'family_history_any', 'family_history_first_degree', 'alcohol_effect_on_tremor']

Modeling table shape: (469, 15)


## 5. Demographic data checks

In [5]:
print("Class distribution:")
display(
    demo_df[TARGET_COLUMN]
    .value_counts()
    .sort_index()
    .rename("count")
    .to_frame()
)

print("\nMissing values:")
display(
    demo_df[demographic_features]
    .isna()
    .sum()
    .sort_values(ascending=False)
    .rename("missing")
    .to_frame()
)

duplicate_ids = demo_df["patient_id"].duplicated(keep=False).sum()
print(f"\nDuplicate participant rows flagged: {duplicate_ids}")

assert duplicate_ids == 0, (
    "Duplicate patient_id values are present. Resolve duplicate participant "
    "records before creating train/validation/test subsets to avoid leakage."
)

Class distribution:


,count
label,
0,79
1,276
2,114



Missing values:


,missing
age_at_diagnosis,98
height_cm,1
age,0
weight_kg,0
gender,0
handedness,0
family_history_any,0
family_history_first_degree,0
alcohol_effect_on_tremor,0



Duplicate participant rows flagged: 0


No imputation or encoding is performed manually. Missing values and feature transformations are handled by the shared preprocessing pipeline during model fitting.

## 6. Create reproducible stratified subsets

In [6]:
# First split: training set versus validation + test remainder.
remaining_size = VALIDATION_SIZE + TEST_SIZE

train_demo, remaining_demo = train_test_split(
    demo_df,
    test_size=remaining_size,
    stratify=demo_df[TARGET_COLUMN],
    random_state=RANDOM_STATE,
)

# Second split: divide the remainder into validation and test sets.
test_fraction_of_remaining = TEST_SIZE / remaining_size

validation_demo, test_demo = train_test_split(
    remaining_demo,
    test_size=test_fraction_of_remaining,
    stratify=remaining_demo[TARGET_COLUMN],
    random_state=RANDOM_STATE,
)

print(f"Train shape:      {train_demo.shape}")
print(f"Validation shape: {validation_demo.shape}")
print(f"Test shape:       {test_demo.shape}")

print("\nParticipant overlap checks:")
print(
    "Train/validation overlap:",
    bool(set(train_demo.patient_id) & set(validation_demo.patient_id)),
)
print(
    "Train/test overlap:",
    bool(set(train_demo.patient_id) & set(test_demo.patient_id)),
)
print(
    "Validation/test overlap:",
    bool(set(validation_demo.patient_id) & set(test_demo.patient_id)),
)

Train shape:      (328, 15)
Validation shape: (70, 15)
Test shape:       (71, 15)

Participant overlap checks:
Train/validation overlap: False
Train/test overlap: False
Validation/test overlap: False


In [7]:
split_distribution = pd.DataFrame({
    "Train": train_demo[TARGET_COLUMN].value_counts().sort_index(),
    "Validation": validation_demo[TARGET_COLUMN].value_counts().sort_index(),
    "Test": test_demo[TARGET_COLUMN].value_counts().sort_index(),
}).fillna(0).astype(int)

split_distribution

,Train,Validation,Test
label,,,
0,55,12,12
1,193,41,42
2,80,17,17


The split proportions and random seed come from `src/modeling/config.py`. Stratification preserves the diagnostic-class distribution across the three subsets.

## 7. Preprocessing documentation and leakage check

In [8]:
X_train, y_train, preprocessing = prepare_dataset(train_demo)

numerical_features, categorical_features = identify_feature_types(X_train)

print("Predictor columns passed to preprocessing:")
print(X_train.columns.tolist())

print("\nNumerical features:")
print(numerical_features)

print("\nCategorical features:")
print(categorical_features)

excluded_check = [
    "patient_id",
    "study_id",
    "duplicate_patient_id",
    TARGET_COLUMN,
    "condition_original",
    "condition_group",
]

remaining_excluded = [
    column
    for column in excluded_check
    if column in X_train.columns
]

assert not remaining_excluded, (
    f"Leakage/identifier columns remain in predictors: {remaining_excluded}"
)

print("\nLeakage prevention check: PASS")
preprocessing

Predictor columns passed to preprocessing:
['age', 'age_at_diagnosis', 'height_cm', 'weight_kg', 'gender', 'handedness', 'family_history_any', 'family_history_first_degree', 'alcohol_effect_on_tremor']

Numerical features:
['age', 'age_at_diagnosis', 'height_cm', 'weight_kg']

Categorical features:
['gender', 'handedness', 'family_history_any', 'family_history_first_degree', 'alcohol_effect_on_tremor']

Leakage prevention check: PASS


ColumnTransformer(transformers=[('numerical',
                                 Pipeline(steps=[('imputer',
                                                  SimpleImputer(strategy='median')),
                                                 ('scaler', StandardScaler())]),
                                 ['age', 'age_at_diagnosis', 'height_cm',
                                  'weight_kg']),
                                ('categorical',
                                 Pipeline(steps=[('imputer',
                                                  SimpleImputer(strategy='most_frequent')),
                                                 ('encoder',
                                                  OneHotEncoder(handle_unknown='ignore'))]),
                                 ['gender', 'handedness', 'family_history_any',
                                  'family_history_first_degree',
                                  'alcohol_effect_on_tremor'])])

The shared preprocessing pipeline:

- median-imputes and standardizes numerical predictors;
- most-frequent-imputes and one-hot encodes categorical predictors;
- excludes identifiers, the target, and diagnosis-related columns before training.

These transformations are fitted inside the model pipeline using the training data.

## 8. Define baseline models

In [18]:
models = {
    "Dummy": get_dummy_classifier(),
    "Logistic Regression": get_logistic_regression(),
    "Random Forest": get_random_forest(),
}

print("Class weighting enabled:", USE_CLASS_WEIGHT)
print("Configured class weight:", CLASS_WEIGHT)
print("Primary metric:", PRIMARY_METRIC)

models

Class weighting enabled: True
Configured class weight: balanced
Primary metric: macro_f1


{'Dummy': DummyClassifier(random_state=42, strategy='most_frequent'),
 'Logistic Regression': LogisticRegression(class_weight='balanced', max_iter=1000, random_state=42),
 'Random Forest': RandomForestClassifier(class_weight='balanced', max_depth=5, min_samples_leaf=5,
                        random_state=42)}

The Dummy Classifier is the reference baseline. Logistic Regression and Random Forest use the class-weight configuration supplied by the shared framework, so the notebook does not override model settings.

## 9. Run the shared modeling workflow

In [10]:
results = run_models(
    models=models,
    train_df=train_demo,
    validation_df=validation_demo,
    test_df=test_demo,
)

print("Completed models:")
print(list(results.keys()))

Completed models:
['Dummy', 'Logistic Regression', 'Random Forest']


## 10. Validation model comparison

In [11]:
validation_comparison = pd.DataFrame({
    model_name: result["metrics"]
    for model_name, result in results.items()
}).T

validation_comparison = validation_comparison.sort_values(
    PRIMARY_METRIC,
    ascending=False,
)

validation_comparison

,accuracy,balanced_accuracy,macro_f1,precision_macro,recall_macro
Random Forest,0.700000,0.740754,0.702882,0.683838,0.740754
Logistic Regression,0.428571,0.422485,0.399894,0.418519,0.422485
Dummy,0.585714,0.333333,0.246246,0.195238,0.333333


## 11. Test model comparison

In [12]:
test_comparison = pd.DataFrame({
    model_name: result["test_metrics"]
    for model_name, result in results.items()
}).T

test_comparison = test_comparison.sort_values(
    PRIMARY_METRIC,
    ascending=False,
)

test_comparison

,accuracy,balanced_accuracy,macro_f1,precision_macro,recall_macro
Random Forest,0.676056,0.665733,0.610798,0.597828,0.665733
Logistic Regression,0.521127,0.490896,0.462041,0.465747,0.490896
Dummy,0.591549,0.333333,0.247788,0.197183,0.333333


## 12. Classification reports

In [13]:
for model_name, result in results.items():
    print("=" * 70)
    print(f"{model_name.upper()} — VALIDATION")
    print("=" * 70)
    display(result["classification_report"])

    print(f"\n{model_name.upper()} — TEST")
    display(result["test_report"])
    print()

DUMMY — VALIDATION


,precision,recall,f1-score,support
0,0.000000,0.000000,0.000000,12.000000
1,0.585714,1.000000,0.738739,41.000000
2,0.000000,0.000000,0.000000,17.000000
accuracy,0.585714,0.585714,0.585714,0.585714
macro avg,0.195238,0.333333,0.246246,70.000000
weighted avg,0.343061,0.585714,0.432690,70.000000



DUMMY — TEST


,precision,recall,f1-score,support
0,0.000000,0.000000,0.000000,12.000000
1,0.591549,1.000000,0.743363,42.000000
2,0.000000,0.000000,0.000000,17.000000
accuracy,0.591549,0.591549,0.591549,0.591549
macro avg,0.197183,0.333333,0.247788,71.000000
weighted avg,0.349931,0.591549,0.439736,71.000000



LOGISTIC REGRESSION — VALIDATION


...

,precision,recall,f1-score,support
0,0.318182,0.583333,0.411765,12.000000
1,0.694444,0.595238,0.641026,42.000000
2,0.384615,0.294118,0.333333,17.000000
accuracy,0.521127,0.521127,0.521127,0.521127
macro avg,0.465747,0.490896,0.462041,71.000000
weighted avg,0.556666,0.521127,0.528605,71.000000



RANDOM FOREST — VALIDATION


,precision,recall,f1-score,support
0,0.733333,0.916667,0.814815,12.0
1,0.818182,0.658537,0.729730,41.0
2,0.500000,0.647059,0.564103,17.0
accuracy,0.700000,0.700000,0.700000,0.7
macro avg,0.683838,0.740754,0.702882,70.0
weighted avg,0.726364,0.700000,0.704092,70.0



RANDOM FOREST — TEST


,precision,recall,f1-score,support
0,0.631579,1.000000,0.774194,12.000000
1,0.761905,0.761905,0.761905,42.000000
2,0.400000,0.235294,0.296296,17.000000
accuracy,0.676056,0.676056,0.676056,0.676056
macro avg,0.597828,0.665733,0.610798,71.000000
weighted avg,0.653225,0.676056,0.652498,71.000000


## 13. Confusion matrices

In [14]:
for model_name, result in results.items():
    print("=" * 70)
    print(f"{model_name.upper()} — VALIDATION CONFUSION MATRIX")
    print("=" * 70)
    display(result["confusion_matrix"])

    print(f"\n{model_name.upper()} — TEST CONFUSION MATRIX")
    display(result["test_confusion_matrix"])
    print()

DUMMY — VALIDATION CONFUSION MATRIX


,Pred_Healthy,Pred_PD,Pred_Other
True_Healthy,0,12,0
True_PD,0,41,0
True_Other,0,17,0



DUMMY — TEST CONFUSION MATRIX


,Pred_Healthy,Pred_PD,Pred_Other
True_Healthy,0,12,0
True_PD,0,42,0
True_Other,0,17,0



LOGISTIC REGRESSION — VALIDATION CONFUSION MATRIX


...

,Pred_Healthy,Pred_PD,Pred_Other
True_Healthy,7,5,0
True_PD,9,25,8
True_Other,6,6,5



RANDOM FOREST — VALIDATION CONFUSION MATRIX


,Pred_Healthy,Pred_PD,Pred_Other
True_Healthy,11,1,0
True_PD,3,27,11
True_Other,1,5,11



RANDOM FOREST — TEST CONFUSION MATRIX


,Pred_Healthy,Pred_PD,Pred_Other
True_Healthy,12,0,0
True_PD,4,32,6
True_Other,3,10,4


## 14. Save demographic modeling outputs

In [15]:
filename_map = {
    "Dummy": "dummy",
    "Logistic Regression": "logistic_regression",
    "Random Forest": "random_forest",
}

for model_name, result in results.items():
    filename = filename_map[model_name]

    # Validation outputs
    save_metrics(
        result["metrics"],
        f"demographics_{filename}_validation_metrics.csv",
    )

    save_classification_report(
        result["classification_report"],
        f"demographics_{filename}_validation_classification_report.csv",
    )

    save_confusion_matrix(
        result["confusion_matrix"],
        f"demographics_{filename}_validation_confusion_matrix.csv",
    )

    save_confusion_matrix_figure(
        result["confusion_matrix"],
        f"demographics_{filename}_validation_confusion_matrix.png",
    )

    # Test outputs
    save_metrics(
        result["test_metrics"],
        f"demographics_{filename}_test_metrics.csv",
    )

    save_classification_report(
        result["test_report"],
        f"demographics_{filename}_test_classification_report.csv",
    )

    save_confusion_matrix(
        result["test_confusion_matrix"],
        f"demographics_{filename}_test_confusion_matrix.csv",
    )

    save_confusion_matrix_figure(
        result["test_confusion_matrix"],
        f"demographics_{filename}_test_confusion_matrix.png",
    )

save_model_comparison(
    validation_comparison,
    "demographics_validation_model_comparison.csv",
)

save_model_comparison(
    test_comparison,
    "demographics_test_model_comparison.csv",
)

save_table(
    split_distribution.reset_index().rename(columns={"index": "label"}),
    "demographics_split_class_distribution.csv",
)

print("Demographic modeling outputs saved successfully.")
print("Metrics/CSV outputs: outputs/metrics/")
print("Confusion-matrix figures: outputs/figures/")

Demographic modeling outputs saved successfully.
Metrics/CSV outputs: outputs/metrics/
Confusion-matrix figures: outputs/figures/


## 15. Initial observations

In [16]:
best_validation_model = validation_comparison.index[0]
best_validation_score = validation_comparison.iloc[0][PRIMARY_METRIC]

best_test_model = test_comparison.index[0]
best_test_score = test_comparison.iloc[0][PRIMARY_METRIC]

dummy_validation_score = validation_comparison.loc[
    "Dummy",
    PRIMARY_METRIC,
]

print("INITIAL DEMOGRAPHIC BASELINE OBSERVATIONS")
print("-" * 45)

print(
    f"Best validation model: {best_validation_model} "
    f"({PRIMARY_METRIC} = {best_validation_score:.3f})"
)

print(
    f"Best test model: {best_test_model} "
    f"({PRIMARY_METRIC} = {best_test_score:.3f})"
)

print(
    f"Dummy validation {PRIMARY_METRIC}: "
    f"{dummy_validation_score:.3f}"
)

if best_validation_score > dummy_validation_score:
    print(
        "At least one demographic model outperformed the Dummy "
        "Classifier on the primary validation metric."
    )
else:
    print(
        "The demographic models did not outperform the Dummy "
        "Classifier on the primary validation metric."
    )

print(
    "Review the class-level reports and confusion matrices above "
    "to identify which diagnostic groups are most frequently confused."
)

INITIAL DEMOGRAPHIC BASELINE OBSERVATIONS
---------------------------------------------
Best validation model: Random Forest (macro_f1 = 0.703)
Best test model: Random Forest (macro_f1 = 0.611)
Dummy validation macro_f1: 0.246
At least one demographic model outperformed the Dummy Classifier on the primary validation metric.
Review the class-level reports and confusion matrices above to identify which diagnostic groups are most frequently confused.


## 16. Final validation status

In [17]:
validation_status = pd.DataFrame({
    "Requirement": [
        "demographics_clean.csv loaded",
        "Demographic-only predictors selected",
        "Reproducible stratified subsets created",
        "No participant overlap across subsets",
        "Shared preprocessing used",
        "Leakage-related columns excluded",
        "Dummy Classifier evaluated",
        "Multinomial Logistic Regression evaluated",
        "Random Forest evaluated",
        "Configured class weighting used where applicable",
        "Validation and test metrics produced",
        "Classification reports produced",
        "Confusion matrices produced",
        "Confusion-matrix figures saved",
        "Outputs saved with shared output utilities",
    ],
    "Status": ["PASS"] * 15,
})

validation_status

,Requirement,Status
0,demographics_clean.csv loaded,PASS
1,Demographic-only predictors selected,PASS
2,Reproducible stratified subsets created,PASS
3,No participant overlap across subsets,PASS
4,Shared preprocessing used,PASS
5,Leakage-related columns excluded,PASS
6,Dummy Classifier evaluated,PASS
7,Multinomial Logistic Regression evaluated,PASS
8,Random Forest evaluated,PASS
9,Configured class weighting used where applicable,PASS


## 17. Conclusion

After successful execution, this notebook provides a demographic-only baseline experiment based directly on `demographics_clean.csv`. It uses the project-wide shared preprocessing, model constructors, workflow, evaluation, and output utilities without duplicating the shared modeling logic.

The updated baseline set consists of the Dummy Classifier, multinomial Logistic Regression, and Random Forest. The resulting validation/test comparisons, class-level reports, confusion matrices, and saved confusion-matrix figures can be used to document the initial predictive value of demographic features relative to the Dummy Classifier.